# tidypr: explore public code-review datasets

**Goal:** see the real columns, sizes, languages and sample rows of the candidate datasets, so the loaders
for the master dataset are written against the actual schema.

Kaggle settings: **Accelerator: None** (CPU is enough, saves GPU hours), **Internet: On**.
Optional: add the `HF_TOKEN` secret (needed only if a dataset is gated).

Run All, then copy everything from the **SUMMARY** cell at the bottom (or the whole output) and paste it back.

In [ ]:
!pip install -q -U huggingface_hub pyarrow pandas
import os, json, re, textwrap, traceback
import pandas as pd, pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
api = HfApi(token=os.environ.get("HF_TOKEN"))
SUMMARY = []
MAX_MB = 400   # don't download anything bigger than this while exploring

def short(v, n=280):
    s = v if isinstance(v, str) else repr(v)
    s = s.replace("\n", "⏎")
    return s if len(s) <= n else s[:n] + f"… (+{len(s)-n} chars)"

def show_files(repo, limit=80):
    try:
        info = api.dataset_info(repo, files_metadata=True)
    except Exception as e:
        print(f"!! {repo}: {type(e).__name__}: {str(e)[:300]}"); SUMMARY.append(f"{repo}: ERROR {type(e).__name__}"); return []
    fs = [(s.rfilename, (s.size or 0) / 1e6) for s in info.siblings]
    print(f"\n### {repo}: {len(fs)} files, gated={getattr(info, 'gated', None)}, license={(info.card_data or {}).get('license') if info.card_data else None}")
    for n, mb in fs[:limit]:
        print(f"  {mb:9.1f} MB  {n}")
    SUMMARY.append(f"{repo}: {len(fs)} files, {sum(m for _, m in fs):.0f} MB total")
    return fs

def peek(repo, fname, n=3, sample_rows=2000):
    try:
        p = hf_hub_download(repo, fname, repo_type="dataset", token=os.environ.get("HF_TOKEN"))
        if fname.endswith(".parquet"):
            pf = pq.ParquetFile(p)
            rows = pf.metadata.num_rows
            df = next(pf.iter_batches(batch_size=sample_rows)).to_pandas()
        elif fname.endswith((".jsonl", ".jsonl.gz")):
            df = pd.read_json(p, lines=True, nrows=sample_rows); rows = "?"
        elif fname.endswith(".json"):
            df = pd.read_json(p).head(sample_rows); rows = "?"
        elif fname.endswith((".csv", ".csv.gz")):
            df = pd.read_csv(p, nrows=sample_rows); rows = "?"
        else:
            print(f"   (skip {fname}: unknown format)"); return None
    except Exception as e:
        print(f"!! {repo}/{fname}: {type(e).__name__}: {str(e)[:300]}"); return None
    print(f"\n--- {repo} / {fname}   rows={rows}")
    print("columns:", {c: str(t) for c, t in df.dtypes.items()})
    for c in df.columns:
        if re.search(r"lang|language|ext|state|agent|type", c, re.I) and df[c].dtype == object:
            vc = df[c].astype(str).value_counts().head(12)
            print(f"  value_counts({c}) [first {len(df)} rows]:", dict(vc))
    for i, row in df.head(n).iterrows():
        print(f"  [row {i}]")
        for c, v in row.items():
            print(f"     {c}: {short(v)}")
    SUMMARY.append(f"{repo}/{fname}: rows={rows} cols={list(df.columns)}")
    return df

## 1. AIDev: review comments on AI-agent PRs

In [ ]:
fs = show_files("hao-li/AIDev")
small = [n for n, mb in fs if mb < MAX_MB and n.endswith((".parquet", ".jsonl", ".csv", ".json", ".gz"))]
for key in ["review_comment", "pr_review", "pull_request", "repository"]:
    for n in [f for f in small if key in f.lower()][:2]:
        aidev_df = peek("hao-li/AIDev", n)

## 2. github-codereview (118K review triplets, 20+ languages)

In [ ]:
fs = show_files("ronantakizawa/github-codereview")
data = [n for n, mb in fs if mb < MAX_MB and n.endswith(".parquet")]
if data:
    gcr_df = peek("ronantakizawa/github-codereview", data[0])
fs = show_files("ronantakizawa/codereview-bench")
data = [n for n, mb in fs if mb < MAX_MB and n.endswith(".parquet")]
for n in data[:2]:
    peek("ronantakizawa/codereview-bench", n)

## 3. Find CodeReviewer (Microsoft) mirrors and other review datasets

In [ ]:
for q in ["codereviewer", "code review", "code-review", "review comment"]:
    print(f"\n### search: {q!r}")
    try:
        for d in api.list_datasets(search=q, sort="downloads", limit=12):
            print(f"  {getattr(d, 'downloads', 0) or 0:>8} dl  {d.id}")
            SUMMARY.append(f"search {q!r}: {d.id} ({getattr(d, 'downloads', 0)} dl)")
    except Exception as e:
        print("  !!", e)

In [ ]:
# peek the most promising CodeReviewer mirrors (edit this list after looking at the search output)
for repo in ["fasterinnerlooper/codereviewer"]:
    fs = show_files(repo)
    data = [n for n, mb in fs if mb < MAX_MB and n.endswith((".parquet", ".jsonl", ".json", ".csv", ".gz"))]
    for n in data[:2]:
        peek(repo, n)

## SUMMARY: copy everything below and paste it back

In [ ]:
print("\n".join(SUMMARY))